# Blender 3D Mesh Rendering

**Purpose:** Uses trimesh to generate 3D mesh renderings of
reconstructed neurons for figure panels.

**Special requirement:** Install Blender
from https://www.blender.org/ to import and visualize the resulting .obj files.

**Inputs:** Either lists of cells or a precomputed format dataset (see Igneous_pipeline)

**Outputs:** .obj files for rendering Figures in Blender

# setup

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import trimesh
import cloudvolume as cv
from cloudvolume import CloudVolume
from cloudvolume.exceptions import MeshDecodeError


In [3]:
import sys
DIR_ROOT = Path.cwd().parent / 'efish_em'
if str(DIR_ROOT) not in sys.path:
    sys.path.insert(0, str(DIR_ROOT))
import AnalysisCode as efish
from eCREST_cli import ecrest

DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'


In [4]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'
vx_sizes = [16, 16, 30]

In [ ]:

vx_sizes = [16,16,30]


In [8]:
efish_cloudvolume = cv.CloudVolume('gs://fish-ell/roi450um_seg32fb16fb_220930', progress=False)


# load file list

In [ ]:
dirpath = DATA_ROOT / 'reconstructions_published'

nodefiles = get_cell_filepaths(dirpath)

# export cells

Either all, single, or list can be specified to render all or subsets of a cell's subcellular compartments

In [ ]:
# Blender .obj exports are written here. Default is a local 'outputs/blender_obj/'
# directory relative to this notebook; override as needed.
savepath = Path.cwd() / 'outputs' / 'blender_obj'
savepath.mkdir(parents=True, exist_ok=True)


In [16]:
all_cells =  [387382792, 472361842, 472284925, 472517114,  40665046, 646634295,
       558300217, 300689181, 300503092, 303166992, 129030308, 472051969]

In [18]:
howmuch = 'list' #'single' #'all'  #
save_str_detail = 'notaxon'
dtype = 'axon'
dtype_list = ['unknown','basal dendrite','apical dendrite','multiple']
reduction_ratio = 0.2

scale_factor = 100000
yaxis_replace_scale = 32768*16/scale_factor
# zaxis_replace_scale = 3534*30/scale_factor
refpt = [0,0,0]

# with tqdm(total=len(all_cells)) as pbar:        
for focal_cell_id in all_cells:
    # pbar.update(1)
    
    dirpath = Path(settings_dict['save_dir'])
    focal_cell_fname = nodefiles[str(focal_cell_id)]#

    cell_data = efish.load_ecrest_celldata(nodefiles[str(focal_cell_id)])
    # cell = ecrest(settings_dict,filepath = dirpath / focal_cell_fname, launch_viewer=False)
    ctype = df_type[df_type['id']==focal_cell_id]['cell_type'].values[0]#cell.get_ctype('manual')

    if howmuch == 'list':
        mesh_ids = [bs for dtype in dtype_list for bs in cell_data['base_segments'][dtype]] # for subset list of cell structures
    if howmuch == 'single':
        mesh_ids = [bs for bs in cell_data['base_segments'][dtype]] # if only need segments from one cell structure #
    if howmuch == 'all':
        mesh_ids = [bs for dtype in cell_data['base_segments'].keys() for bs in cell_data['base_segments'][dtype]] #if need all segments

    # grab the mesh for more than one object
    # Initialize an empty Trimesh object
    fused_mesh = trimesh.Trimesh()
    with tqdm(total=len(mesh_ids)) as pbar:        
        for id in mesh_ids:
            pbar.update(1)
            try:
                mesh = efish_cloudvolume.mesh.get(int(id))
                vertices = list(mesh.values())[0].vertices
                vertices = vertices/scale_factor
                vertices = [[v[0]-refpt[0],v[1]-refpt[1],v[2]-refpt[2]] for i,v in enumerate(vertices)]
                vertices = [[v[0],yaxis_replace_scale-v[1],v[2]] for v in vertices]
                faces = list(mesh.values())[0].faces
                trimesh_mesh = trimesh.Trimesh(vertices=vertices, faces=faces,process=True)
                
                # downsample the fused mesh before joining to the fused_mesh and exporting as a .obj for blender
                simplified_mesh = trimesh_mesh.simplify_quadric_decimation(face_count=len(trimesh_mesh.vertices) * reduction_ratio)
                fused_mesh = trimesh.util.concatenate([fused_mesh, simplified_mesh])
  
            except MeshDecodeError as e: 
                print(f"An unexpected error occurred on segment{id}: {e}")
                continue

            except ValueError as e:
                print(f"An unexpected error occurred on segment{id}: {e}")
                continue
    
    fused_mesh.export(savepath / f'{ctype}_{focal_cell_id}_{save_str_detail}.obj');

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2179/2179 [11:19<00:00,  3.21it/s]


# OBJ from CloudVolume retrieve http

This can be used for an igneous-view served local http url OR for the public segmentation bucket

In [48]:
vol = CloudVolume('precomputed://http://localhost:8001')

In [49]:
segIDs_all = vol.unique(bbox=vol.bounds) - set([0]) # get all segment IDs in volume and remove segment ID 0, which is background

Downloading: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 572/572 [00:04<00:00, 122.06it/s]


In [ ]:
# Local CloudVolume mesh .obj exports are written here. Default is a local
# 'outputs/blender_obj_local_cv/' directory relative to this notebook; override as needed.
savepath = Path.cwd() / 'outputs' / 'blender_obj_local_cv'
savepath.mkdir(parents=True, exist_ok=True)


for s in list(segIDs_all):
        
    mesh= vol.mesh.get(int(s))
    
    # reduction_ratio = 0.2
    
    scale_factor = 100000
    yaxis_replace_scale = 32768*16/scale_factor
    # zaxis_replace_scale = 3534*30/scale_factor
    refpt = [0,0,0]
    
    vertices = mesh.vertices
    vertices = vertices/scale_factor
    vertices = [[v[0]-refpt[0],v[1]-refpt[1],v[2]-refpt[2]] for i,v in enumerate(vertices)]
    vertices = [[v[0],yaxis_replace_scale-v[1],v[2]] for v in vertices]
    faces = mesh.faces
    trimesh_mesh = trimesh.Trimesh(vertices=vertices, faces=faces,process=True)

    # downsample the fused mesh before joining to the fused_mesh and exporting as a .obj for blender
    # simplified_mesh = trimesh_mesh.simplify_quadric_decimation(face_count=len(trimesh_mesh.vertices) * reduction_ratio)
    
    
    trimesh_mesh.export(savepath / f'{s}.obj')